In [ ]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/Projects_AI_DS/bearing_fault/IMS.zip"
extract_path = "/content/mydata"

# Create folder if not exists
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted files:", os.listdir(extract_path))


Extracted files: ['IMS', '__MACOSX']


In [ ]:
!apt-get install unrar
!unrar x /content/mydata/IMS/1st_test.rar /content/mydata/IMS/
# !unrar x /content/mydata/IMS/2nd_test.rar /content/mydata/IMS/
# !unrar x /content/mydata/IMS/3rd_test.rar /content/mydata/IMS/

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from /content/mydata/IMS/1st_test.rar

Creating    /content/mydata/IMS/1st_test                              OK
Extracting  /content/mydata/IMS/1st_test/2003.10.22.12.06.24               0%  OK 
Extracting  /content/mydata/IMS/1st_test/2003.10.22.12.09.13               0%  OK 
Extracting  /content/mydata/IMS/1st_test/2003.10.22.12.14.13               0%  OK 
Extracting  /content/mydata/IMS/1st_test/2003.10.22.12.19.13               0%  OK 
Extracting  /content/mydata/IMS/1st_test/2003.10.22.12.24.13               0%  OK 
Extracting  /content/mydata/IMS/1st_test/2003.10.22.12.29.13               0%  OK 
Extracting  /content/mydata/IMS

buffer_zone

In [ ]:
# ================= IMS CLASSIFICATION (LABEL FIXED, SAME PROBLEM) =================

import os
import numpy as np
import pandas as pd

from scipy.stats import skew, kurtosis
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix
from lightgbm import LGBMClassifier

# ---------------- CONFIG ----------------
data_dir = "/content/mydata/IMS/1st_test"

window_size = 2048
stride = 1024

HEALTHY_END = int(0.6 * len(os.listdir(data_dir)))   # stable healthy region
FAULT_START = int(0.75 * len(os.listdir(data_dir)))  # true fault region


# ---------------- WINDOWING ----------------
def get_windows(sig):
    return np.array([
        sig[i:i+window_size]
        for i in range(0, len(sig) - window_size, stride)
    ])


# ---------------- FEATURE ENGINEERING ----------------
def extract_window_features(windows):

    feats = []

    for ch in range(windows.shape[2]):
        x = windows[:, :, ch]
        t = np.arange(x.shape[1])

        rms = np.sqrt(np.mean(x**2, axis=1))
        std = np.std(x, axis=1)
        sk = skew(x, axis=1)
        ku = np.nan_to_num(kurtosis(x, axis=1))
        env = np.mean(np.abs(x), axis=1)

        crest = np.max(np.abs(x), axis=1) / (rms + 1e-6)

        fft = np.abs(np.fft.rfft(x, axis=1))
        hf_ratio = np.sum(fft[:, 50:]**2, axis=1) / (np.sum(fft**2, axis=1) + 1e-6)

        slope = np.array([np.polyfit(t, x[i], 1)[0] for i in range(len(x))])

        feats.append(np.stack([
            rms, std, sk, ku, env,
            crest,
            hf_ratio,
            slope
        ], axis=1))

    return np.concatenate(feats, axis=1)


# ---------------- BUILD DATASET ----------------
files = sorted(os.listdir(data_dir))

X, y, groups = [], [], []

for i, f in enumerate(files):

    sig = pd.read_csv(
        os.path.join(data_dir, f),
        sep=r'\s+',
        header=None
    ).values

    B3 = sig[:, 4:6]
    B4 = sig[:, 6:8]

    w3 = get_windows(B3)
    w4 = get_windows(B4)

    if len(w3) == 0 or len(w4) == 0:
        continue

    f3 = extract_window_features(w3)
    f4 = extract_window_features(w4)

    # ---------------- CORRECT LABELING ----------------
    if i < HEALTHY_END:
        y3 = np.zeros(len(f3))
        y4 = np.zeros(len(f4))

    elif i < FAULT_START:
        # transition region → keep as healthy (IMPORTANT FIX)
        y3 = np.zeros(len(f3))
        y4 = np.zeros(len(f4))

    else:
        y3 = np.ones(len(f3))      # inner race fault (Class 1)
        y4 = np.ones(len(f4)) * 2  # ball fault (Class 2)

    X.append(f3)
    y.append(y3)
    groups.append(np.full(len(f3), i))

    X.append(f4)
    y.append(y4)
    groups.append(np.full(len(f4), i))


X = np.vstack(X)
y = np.concatenate(y)
groups = np.concatenate(groups)

print("Dataset:", X.shape)
print("Classes:", np.unique(y, return_counts=True))


# ---------------- GROUP SPLIT ----------------
gss = GroupShuffleSplit(test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]


# ---------------- SCALING ----------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# ---------------- MODEL ----------------
model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=64,
    class_weight={0:1, 1:2, 2:1},
    random_state=42
)

model.fit(X_train, y_train)


# ---------------- EVALUATION ----------------
pred = model.predict(X_test)

print("\nWINDOW RESULT:")
print(classification_report(y_test, pred))

print("\nCONFUSION MATRIX:")
print(confusion_matrix(y_test, pred))

Dataset: (77616, 16)
Classes: (array([0., 1., 2.]), array([58212,  9702,  9702]))
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009683 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4080
[LightGBM] [Info] Number of data points in the train set: 54324, number of used features: 16
[LightGBM] [Info] Start training from score -0.399290
[LightGBM] [Info] Start training from score -1.516544
[LightGBM] [Info] Start training from score -2.209691


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



WINDOW RESULT:
              precision    recall  f1-score   support

         0.0       0.94      0.97      0.95     17280
         1.0       0.77      0.64      0.70      3006
         2.0       0.94      0.96      0.95      3006

    accuracy                           0.92     23292
   macro avg       0.88      0.86      0.87     23292
weighted avg       0.92      0.92      0.92     23292


CONFUSION MATRIX:
[[16693   507    80]
 [  974  1917   115]
 [   52    62  2892]]


In [ ]:
# ================= IMS FINAL TUNED MODEL (SAFE LAST BOOST) =================

import os
import numpy as np
import pandas as pd

from scipy.stats import skew, kurtosis
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix
from lightgbm import LGBMClassifier

# ---------------- CONFIG ----------------
data_dir = "/content/mydata/IMS/1st_test"

window_size = 2048
stride = 1024

files = sorted(os.listdir(data_dir))
N = len(files)

HEALTHY_END = int(0.6 * N)
FAULT_START = int(0.75 * N)


# ---------------- WINDOWING ----------------
def get_windows(sig):
    return np.array([
        sig[i:i+window_size]
        for i in range(0, len(sig) - window_size, stride)
    ])


# ---------------- FEATURE ENGINEERING (YOUR BEST STABLE SET) ----------------
def extract_window_features(windows):

    feats = []

    for ch in range(windows.shape[2]):
        x = windows[:, :, ch]
        t = np.arange(x.shape[1])

        rms = np.sqrt(np.mean(x**2, axis=1))
        std = np.std(x, axis=1)
        sk = skew(x, axis=1)
        ku = np.nan_to_num(kurtosis(x, axis=1))
        env = np.mean(np.abs(x), axis=1)

        crest = np.max(np.abs(x), axis=1) / (rms + 1e-6)

        fft = np.abs(np.fft.rfft(x, axis=1))
        hf_ratio = np.sum(fft[:, 50:]**2, axis=1) / (np.sum(fft**2, axis=1) + 1e-6)

        slope = np.array([np.polyfit(t, x[i], 1)[0] for i in range(len(x))])

        feats.append(np.stack([
            rms, std, sk, ku,
            env, crest,
            hf_ratio,
            slope
        ], axis=1))

    return np.concatenate(feats, axis=1)


# ---------------- BUILD DATASET ----------------
X, y, groups = [], [], []

for i, f in enumerate(files):

    sig = pd.read_csv(
        os.path.join(data_dir, f),
        sep=r'\s+',
        header=None
    ).values

    B3 = sig[:, 4:6]
    B4 = sig[:, 6:8]

    w3 = get_windows(B3)
    w4 = get_windows(B4)

    if len(w3) == 0 or len(w4) == 0:
        continue

    f3 = extract_window_features(w3)
    f4 = extract_window_features(w4)

    # ---------------- FINAL CORRECT LABELING ----------------
    if i < HEALTHY_END:
        y3 = np.zeros(len(f3))
        y4 = np.zeros(len(f4))

    elif i < FAULT_START:
        y3 = np.zeros(len(f3))   # transition kept as healthy (stable version)
        y4 = np.zeros(len(f4))

    else:
        y3 = np.ones(len(f3))      # Class 1 (inner race)
        y4 = np.ones(len(f4)) * 2  # Class 2 (ball fault)

    X.append(f3)
    y.append(y3)
    groups.append(np.full(len(f3), i))

    X.append(f4)
    y.append(y4)
    groups.append(np.full(len(f4), i))


X = np.vstack(X)
y = np.concatenate(y)
groups = np.concatenate(groups)

print("Dataset:", X.shape)
print("Classes:", np.unique(y, return_counts=True))


# ---------------- SPLIT (LEAKAGE SAFE) ----------------
gss = GroupShuffleSplit(test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]


# ---------------- SCALING ----------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# ---------------- FINAL TUNED MODEL ----------------
model = LGBMClassifier(
    n_estimators=700,
    learning_rate=0.025,

    num_leaves=80,
    max_depth=-1,

    min_child_samples=40,
    subsample=0.9,
    colsample_bytree=0.9,

    reg_alpha=0.1,
    reg_lambda=0.1,

    class_weight={0:1, 1:2.2, 2:1},

    random_state=42
)

model.fit(X_train, y_train)


# ---------------- EVALUATION ----------------
pred = model.predict(X_test)

print("\nWINDOW RESULT:")
print(classification_report(y_test, pred))

print("\nCONFUSION MATRIX:")
print(confusion_matrix(y_test, pred))

Dataset: (77616, 16)
Classes: (array([0., 1., 2.]), array([58212,  9702,  9702]))
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009916 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4080
[LightGBM] [Info] Number of data points in the train set: 54324, number of used features: 16
[LightGBM] [Info] Start training from score -0.420999
[LightGBM] [Info] Start training from score -1.442944
[LightGBM] [Info] Start training from score -2.231401


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



WINDOW RESULT:
              precision    recall  f1-score   support

         0.0       0.94      0.97      0.95     17280
         1.0       0.78      0.64      0.70      3006
         2.0       0.94      0.96      0.95      3006

    accuracy                           0.92     23292
   macro avg       0.89      0.86      0.87     23292
weighted avg       0.92      0.92      0.92     23292


CONFUSION MATRIX:
[[16724   476    80]
 [  978  1917   111]
 [   52    61  2893]]


In [ ]:
# ================= IMS FULL PIPELINE (PHYSICS-BASED LABELING) =================

import os
import numpy as np
import pandas as pd

from scipy.stats import kurtosis, skew
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix
from lightgbm import LGBMClassifier

# ---------------- CONFIG ----------------
data_dir = "/content/mydata/IMS/1st_test"
files = sorted(os.listdir(data_dir))


# =========================================================
# STEP 1: PHYSICS-BASED FAULT ONSET DETECTION (LABELING)
# =========================================================

rms_series = []
kurt_series = []
env_series = []

for f in files:

    sig = pd.read_csv(
        os.path.join(data_dir, f),
        sep=r'\s+',
        header=None
    ).values

    # B3 (inner race bearing used for degradation detection)
    B3 = sig[:, 4:6]

    rms = np.sqrt(np.mean(B3**2))
    kur = kurtosis(B3.flatten())
    env = np.mean(np.abs(B3))

    rms_series.append(rms)
    kurt_series.append(kur)
    env_series.append(env)

rms_series = np.array(rms_series)
kurt_series = np.array(kurt_series)
env_series = np.array(env_series)

# normalize signals
features = np.stack([rms_series, kurt_series, env_series], axis=1)
features = StandardScaler().fit_transform(features)

# degradation signal = combined change rate
grad = np.sum(np.abs(np.gradient(features, axis=0)), axis=1)

smooth = pd.Series(grad).rolling(15).mean().fillna(0)

fault_start = np.argmax(smooth.values)
buffer = int(0.05 * len(files))

HEALTHY_END = max(0, fault_start - buffer)
FAULT_START = fault_start

print("Physics-based HEALTHY_END:", HEALTHY_END)
print("Physics-based FAULT_START:", FAULT_START)


# =========================================================
# STEP 2: WINDOWING
# =========================================================

window_size = 2048
stride = 1024

def get_windows(sig):
    return np.array([
        sig[i:i+window_size]
        for i in range(0, len(sig) - window_size, stride)
    ])


# =========================================================
# STEP 3: FEATURE ENGINEERING
# =========================================================

def extract_features(windows):

    feats = []

    for ch in range(windows.shape[2]):
        x = windows[:, :, ch]
        t = np.arange(x.shape[1])

        rms = np.sqrt(np.mean(x**2, axis=1))
        std = np.std(x, axis=1)
        sk = skew(x, axis=1)
        ku = np.nan_to_num(kurtosis(x, axis=1))
        env = np.mean(np.abs(x), axis=1)

        crest = np.max(np.abs(x), axis=1) / (rms + 1e-6)

        fft = np.abs(np.fft.rfft(x, axis=1))
        hf_ratio = np.sum(fft[:, 50:]**2, axis=1) / (np.sum(fft**2, axis=1) + 1e-6)

        slope = np.array([np.polyfit(t, x[i], 1)[0] for i in range(len(x))])

        feats.append(np.stack([
            rms, std, sk, ku,
            env, crest,
            hf_ratio,
            slope
        ], axis=1))

    return np.concatenate(feats, axis=1)


# =========================================================
# STEP 4: BUILD DATASET (LABELING USING PHYSICS THRESHOLD)
# =========================================================

X, y, groups = [], [], []

for i, f in enumerate(files):

    sig = pd.read_csv(
        os.path.join(data_dir, f),
        sep=r'\s+',
        header=None
    ).values

    B3 = sig[:, 4:6]
    B4 = sig[:, 6:8]

    w3 = get_windows(B3)
    w4 = get_windows(B4)

    if len(w3) == 0 or len(w4) == 0:
        continue

    f3 = extract_features(w3)
    f4 = extract_features(w4)

    # ---------------- PHYSICS-BASED LABELING ----------------
    if i < HEALTHY_END:
        y3 = np.zeros(len(f3))
        y4 = np.zeros(len(f4))

    elif i < FAULT_START:
        y3 = np.zeros(len(f3))   # transition treated as healthy (clean baseline)
        y4 = np.zeros(len(f4))

    else:
        y3 = np.ones(len(f3))      # inner race fault
        y4 = np.ones(len(f4)) * 2  # ball fault

    X.append(f3)
    y.append(y3)
    groups.append(np.full(len(f3), i))

    X.append(f4)
    y.append(y4)
    groups.append(np.full(len(f4), i))


X = np.vstack(X)
y = np.concatenate(y)
groups = np.concatenate(groups)

print("\nDataset:", X.shape)
print("Classes:", np.unique(y, return_counts=True))


# =========================================================
# STEP 5: LEAKAGE-FREE SPLIT
# =========================================================

gss = GroupShuffleSplit(test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]


# =========================================================
# STEP 6: SCALING
# =========================================================

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# =========================================================
# STEP 7: MODEL
# =========================================================

model = LGBMClassifier(
    n_estimators=600,
    learning_rate=0.03,
    num_leaves=64,
    class_weight={0:1, 1:2.2, 2:1},
    random_state=42
)

model.fit(X_train, y_train)


# =========================================================
# STEP 8: EVALUATION
# =========================================================

pred = model.predict(X_test)

print("\nWINDOW RESULT:")
print(classification_report(y_test, pred))

print("\nCONFUSION MATRIX:")
print(confusion_matrix(y_test, pred))

Physics-based HEALTHY_END: 2048
Physics-based FAULT_START: 2155

Dataset: (77616, 16)
Classes: (array([0., 1., 2.]), array([77580,    18,    18]))
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009277 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4080
[LightGBM] [Info] Number of data points in the train set: 54324, number of used features: 16
[LightGBM] [Info] Start training from score -0.001060
[LightGBM] [Info] Start training from score -7.224290
[LightGBM] [Info] Start training from score -8.012747
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fur

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



WINDOW RESULT:
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00     23292
         1.0       0.00      0.00      0.00         0
         2.0       0.00      0.00      0.00         0

    accuracy                           1.00     23292
   macro avg       0.33      0.33      0.33     23292
weighted avg       1.00      1.00      1.00     23292


CONFUSION MATRIX:
[[23278     4    10]
 [    0     0     0]
 [    0     0     0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
